In [1]:
import torch
from dinosaw.utils import get_features, add_custom_font, do_2D_pca
from dinosaw.wrappers import ModelTypes, MODEL_NAMES, get_models

import numpy as np
from tifffile import imread
from PIL import Image

from skimage.transform import resize

import matplotlib.pyplot as plt
from typing import TypeAlias, Literal

from interactive_seg_backend import TrainingConfig, FeatureConfig, featurise, concat_feats, train_and_apply
from interactive_seg_backend.file_handling import load_labels

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-07-24 10:10:36 | I | multiscale_classical_cpu.py:  46 | N CPUS: 110
2026-07-24 10:10:36 | W | gpu_utils.py               :  21 | CuPY not installed, GPU fit/apply unavailable!


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dvt_dinov2_s', 'alibi_coco_dinov2_s', 'dinov3_s+', 'alibi_dinov3_s+_norm_wrap_ms')
models = get_models(enabled_models, DEVICE, True, "../../models/checkpoints", "../../models/dinov3")

f_cfg = FeatureConfig()

tr_cfg = TrainingConfig(feature_config=f_cfg, CRF=True, classifier='xgb', CRF_params={"label_confidence": 0.6,
    "sxy_g": [1, 1],
    "sxy_b": [30, 30],
    "s_rgb": [13, 13, 13],
    "compat_g": 10,
    "compat_b": 10,
    "n_infer": 10}, classifier_params={"class_weight": "balanced", "max_depth": 8}) 

2026-07-24 10:10:39 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 10:10:39 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 10:10:39 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 10:10:39 | I | factory.py                 : 152 | Building wrapper 'dvt_dinov2_s' on device cuda:0
2026-07-24 10:10:39 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_p

In [3]:
ds_folder = '../paper_figures/data/trainable_seg'

AllowedImages = Literal['anode', 'curtains', 'bimodal', 'NMC', 'steel', 'rome', 'butterflies', 'au_foil']


side_lengths: dict[AllowedImages, int] = {
    'au_foil': 737,
    'anode': 756,
    'curtains': 756,
    'rome': 756,
    'butterflies': 756,
    'bimodal': 839,
    'NMC': 602,
    'steel': 602,
    'pre_kint': 1024,
}


image_fnames: list[AllowedImages] = ['rome', 'curtains', 'anode']
# image_fnames: list[AllowedImages] = ['curtains', 'anode', 'pre_kint']
images: list[Image.Image] = []
labels: list[np.ndarray] = []
for image_fname in image_fnames:
    img = Image.open(f"{ds_folder}/{image_fname}.png").convert("RGB")
    shortest = min(img.size)
    L = side_lengths[image_fname]
    sf = L / shortest
    new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    img = img.resize(new_size, resample=Image.BILINEAR)
    images.append(img)

    label = load_labels(f"{ds_folder}/{image_fname}_biased_labels.tiff")
    labels.append(label)

In [4]:
features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}
red_features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}


for key, model in models.items():
    for img in images:

        classical = featurise(np.array(img.convert('L')), tr_cfg)

        if key == 'classical':
            features[key].append(classical)
            red_features[key].append(classical)
            continue

        feats = get_features(model, img, False)
        reduced = do_2D_pca(feats, 9, pre_norm='std', post_norm='minmax')
        reduced_hr = resize(reduced, (img.height, img.width), order=1, mode='reflect', anti_aliasing=True)

        combined_feats = concat_feats(reduced_hr, classical)
        features[key].append(combined_feats)

        red_features[key].append(reduced_hr)

2026-07-24 10:10:40 | I | multiscale_classical_cpu.py: 466 | CPU feats on (756, 1008) with `default`: weka-style features
2026-07-24 10:10:41 | I | main.py                    :  71 | Features out: (756, 1008, 58)
2026-07-24 10:10:41 | I | wrapper.py                 :  92 | Processing image, size: [1008, 756]
2026-07-24 10:10:42 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,756,1008] -> f: [1,384,54,72]
2026-07-24 10:10:42 | I | multiscale_classical_cpu.py: 466 | CPU feats on (756, 1047) with `default`: weka-style features
2026-07-24 10:10:43 | I | main.py                    :  71 | Features out: (756, 1047, 58)
2026-07-24 10:10:43 | I | wrapper.py                 :  92 | Processing image, size: [1047, 756]
2026-07-24 10:10:43 | I | wrapper.py                 : 192 | Forward Features: x: [1,3,756,1036] -> f: [1,384,54,74]
2026-07-24 10:10:43 | I | multiscale_classical_cpu.py: 466 | CPU feats on (756, 1319) with `default`: weka-style features
2026-07-24 10:10:45 | I 

In [5]:
preds: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}

for key, model in models.items():
    for i, feat in enumerate(features[key]):
        image = np.array(images[i])
        label = labels[i]

        pred, _, _ = train_and_apply(feat, label, tr_cfg, image=image)
        preds[key].append(pred)

2026-07-24 10:11:04 | I | core.py                    : 155 | Training XGBClassifier: (12166, 67) -> ((12166,)) 
2026-07-24 10:11:05 | I | core.py                    : 170 | Applying XGBClassifier to (756, 1008, 67) features


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [10:11:04] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 10:11:05 | I | crf.py                     :  77 | CRF: probs (756, 1008, 4), img (756, 1008, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-07-24 10:11:06 | I | core.py                    : 155 | Training XGBClassifier: (4182, 67) -> ((4182,)) 
2026-07-24 10:11:06 | I | core.py                    : 170 | Applying XGBClassifier to (756, 1047, 67) features
2026-07-24 10:11:06 | I | crf.py                     :  77 | CRF: probs (756, 1047, 4), img (756, 1047, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [10:11:06] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 10:11:07 | I | core.py                    : 155 | Training XGBClassifier: (2928, 67) -> ((2928,)) 
2026-07-24 10:11:07 | I | core.py                    : 170 | Applying XGBClassifier to (756, 1319, 67) features
2026-07-24 10:11:07 | I | crf.py                     :  77 | CRF: probs (756, 1319, 3), img (756, 1319, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


...

2026-07-24 10:11:20 | I | crf.py                     :  77 | CRF: probs (756, 1008, 4), img (756, 1008, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-07-24 10:11:21 | I | core.py                    : 155 | Training XGBClassifier: (4182, 67) -> ((4182,)) 
2026-07-24 10:11:21 | I | core.py                    : 170 | Applying XGBClassifier to (756, 1047, 67) features


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [10:11:21] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 10:11:21 | I | crf.py                     :  77 | CRF: probs (756, 1047, 4), img (756, 1047, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-07-24 10:11:22 | I | core.py                    : 155 | Training XGBClassifier: (2928, 67) -> ((2928,)) 
2026-07-24 10:11:22 | I | core.py                    : 170 | Applying XGBClassifier to (756, 1319, 67) features


/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [10:11:22] WARNING: /workspace/src/learner.cc:793: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-24 10:11:22 | I | crf.py                     :  77 | CRF: probs (756, 1319, 3), img (756, 1319, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


In [6]:
from skimage.color import label2rgb
from PIL.ImageColor import getcolor

COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]



# add_custom_font('resources/fonts', 'Grotesk')

def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])



def apply_labels_as_overlay(labels: np.ndarray, img: Image.Image, colors: list, alpha: float=1.0) -> Image.Image:
    labels_unsqueezed = np.expand_dims(labels, -1)

    overlay = label2rgb(labels, colors=colors[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=alpha)
    out = np.where(labels_unsqueezed, overlay * 255, np.array(img)).astype(np.uint8)
    img_with_labels = Image.fromarray(out)
    return img_with_labels

In [ ]:
# %%capture
plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')
# W, H = 7, 2.3 * 3.25
W, H = 7.5, 2.3 * 4

N_ROWS = len(enabled_models) + 1
N_COLS = len(image_fnames)
fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(W, H))

titles = ["Bimodal cathode", "Anode", "Cathode (+CC)"]

for y in range(N_ROWS):
    for x in range(N_COLS):
        if y == 0:
            axs[y, 0].set_ylabel("Image + labels")
            axs[y, x].set_title(titles[x])
            arr = apply_labels_as_overlay(labels[x], images[x], COLORS, alpha=1)
        else:
            key = enabled_models[y-1]
            model_name = MODEL_NAMES[key]
            model_name = model_name.replace('(COCO)', '')
            weight = 700 if 'alibi' in key.lower() else 500
            axs[y, 0].set_ylabel(model_name, weight=weight)
            arr = preds[key][x]
            arr = label2rgb(arr + 1, colors=COLORS[1:], bg_label=0)
        axs[y, x].imshow(arr, aspect='auto', interpolation='nearest')
        hide_axis(axs[y, x])

SAVE = True
if SAVE:
    plt.savefig('saved/09.pdf', dpi=300, bbox_inches='tight')
    plt.close()


findfont: Failed to find font weight normal, now using 300.


findfont: Failed to find font weight 500, now using 300.
